# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YosraHilmi/Fly-Rank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

one row = one page (content) per month. The source table, fact_content_daily_performance, has one row per day × client × content, so the daily rows of each page are aggregated into a single row: impressions and clicks are summed, CTR is recomputed from the totals (total clicks ÷ total impressions × 100), and average position is weighted by impressions

time window: one month (march 3-2026), each row represent page actvities over that month.

Chosen as a mid-panel month: the first month of the data is partial and the last month (June 2026) is excluded to avoid revealing the outcome window.


---




## 2. Fields: feature / label / context / excluded

Tables: fact_content_daily performance (performance), joined to dim_content on client hash id + content hash id (page attributes).

Features: avg position (computed as SUM(gs_sum_position) + SUM(gs_impressions) over the month, to be verified against gs_avg_position), content_type, main_intent.

Label: CTR = SUM(gs_clicks) + SUM(gs_impressions) × 100 (no CTR column exists in the warehouse).

Context: content_hash id (with client_hash_id), gs_impressions, last_optimized_date. Deferred: pc (to be tested later).

execluded:

clicks (gs_clicks), clicks_last_30d: the label is computed from them, so the model could recompute the label instead of learning from page characteristics (leakage).

ctr: it is the label itself.

impressions _last_ 30d: a sub-window of impressions, which is kept as context, not a feature (volume is already used in the score).

Analytics columns (sessions, engagement _rate, scroll rate, ai sessions, ai raffic pet): they happen after the click, so they are a result of CTR, not a cause.

---




## 3. Verify it with queries (grain, counts, missing values, windows)


In [6]:

from google.colab import userdata
import duckdb

# Retrieve the actual token value
hf_token = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Create the Hugging Face secret using the actual token
con.execute("""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN $token
    )
""", {"token": hf_token})

# Query the dataset
rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").fetchone()

print("Total rows:", result[0])



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 78835655


In [7]:

# Cell 0 - Setup
MONTH = "2026-03"
base = f"{rel}/fact_content_daily_performance"
march_path = f"{base}/month={MONTH}/**/*.parquet"

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret
(TYPE huggingface, TOKEN $token)
""", {"token": hf_token})

# Try the recursive March-only partition path first.
try:
    con.execute(f"""
    CREATE OR REPLACE VIEW fact_mar AS
    SELECT * FROM read_parquet(
        '{march_path}', hive_partitioning=true
    )
    """)
    print("Working fact pattern: March partition recursive glob")
except Exception as e:
    print("First fact pattern failed:", str(e)[:500])
    try:
        march_path = f"{base}/month={MONTH}/*.parquet"
        con.execute(f"""
        CREATE OR REPLACE VIEW fact_mar AS
        SELECT * FROM read_parquet(
            '{march_path}', hive_partitioning=true
        )
        """)
        print("Working fact pattern: March partition direct glob")
    except Exception as e2:
        print("Second fact pattern failed:", str(e2)[:500])
        raise RuntimeError("Could not read the March partition.")

# Try dim_content as a Parquet folder, then as a single Parquet file.
dim_folder = f"{rel}/dim_content/**/*.parquet"
try:
    con.execute(f"""
    CREATE OR REPLACE TABLE dim_content_v AS
    SELECT client_hash_id, content_hash_id, content_type,
           main_intent, is_published, is_deleted
    FROM read_parquet('{dim_folder}')
    """)
    print("Working dim_content pattern: folder Parquet files")
except Exception as e:
    print("First dim_content pattern failed:", str(e)[:500])
    dim_file = f"{rel}/dim_content.parquet"
    try:
        con.execute(f"""
        CREATE OR REPLACE TABLE dim_content_v AS
        SELECT client_hash_id, content_hash_id, content_type,
               main_intent, is_published, is_deleted
        FROM read_parquet('{dim_file}')
        """)
        print("Working dim_content pattern: single Parquet file")
    except Exception as e2:
        print("Second dim_content pattern failed:", str(e2)[:500])
        raise RuntimeError("Could not read dim_content.")

print("setup ok")


Working fact pattern: March partition recursive glob
First dim_content pattern failed: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_content' (HTTP 404)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working dim_content pattern: single Parquet file
setup ok


**1- one row per day x client x content**

In [8]:

# Cell 1 - First 5 rows of the daily table
display(con.sql("""
    SELECT report_date, content_hash_id, gsc_impressions,
           gsc_clicks, gsc_avg_position AS avg_position
    FROM fact_mar
    LIMIT 5
""").df())

print("Grain = one row per report_date x client x content (the same page appears on many days).")


,report_date,content_hash_id,gsc_impressions,gsc_clicks,avg_position
0,2026-03-01,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,content_a3ea9792f793ec72,11,0,2.272727


Grain = one row per report_date x client x content (the same page appears on many days).


In [9]:

# Cell 2 - Page-month table
con.execute("""
CREATE OR REPLACE TABLE page_month AS
WITH page_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_clicks) * 100.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_sum_position)
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM fact_mar
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    p.*,
    d.content_type,
    d.main_intent,
    d.is_published,
    d.is_deleted,
    (d.content_hash_id IS NOT NULL) AS matched_in_dim
FROM page_agg p
LEFT JOIN dim_content_v d
    ON p.client_hash_id = d.client_hash_id
   AND p.content_hash_id = d.content_hash_id
""")

display(con.sql("""
    SELECT content_hash_id, total_impressions, total_clicks, ctr,
           avg_position, content_type, main_intent,
           is_published, is_deleted
    FROM page_month
    LIMIT 5
""").df())

before_join = con.sql("""
    SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id
        FROM fact_mar
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
""").fetchone()[0]

after_join = con.sql("SELECT COUNT(*) FROM page_month").fetchone()[0]
raw_rows = con.sql("SELECT COUNT(*) FROM fact_mar").fetchone()[0]
excluded = con.sql("""
    SELECT COUNT(*) FROM fact_mar
    WHERE gsc_data_available IS NOT TRUE
""").fetchone()[0]
zero_impressions = con.sql("""
    SELECT COUNT(*) FROM page_month WHERE total_impressions = 0
""").fetchone()[0]

print("Rows before join:", before_join)
print("Rows after join:", after_join)
print("Join preserved row count (no row multiplication):", before_join == after_join)
print("Raw daily rows excluded by gsc_data_available IS TRUE:", excluded)
print("Excluded row percentage:", round(100 * excluded / raw_rows, 2) if raw_rows else None)
print("Pages with total_impressions = 0 (CTR remains NULL):", zero_impressions)

print("Calculation notes:")
print("1. Impressions and clicks are summed over the month.")
print("2. CTR = monthly total clicks / monthly total impressions x 100, not an average of daily CTRs; high-volume days therefore contribute proportionally more.")
print("3. Average position = SUM(gsc_sum_position) / SUM(gsc_impressions); days with more impressions carry more weight.")
print("4. Only rows where gsc_data_available IS TRUE were used.")
print("5. dim_content was joined with LEFT JOIN on client_hash_id + content_hash_id; unmatched pages are retained and counted.")
print("6. is_published and is_deleted are displayed, not used as filters; these fields describe the July 2026 export, not March 2026.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions,total_clicks,ctr,avg_position,content_type,main_intent,is_published,is_deleted
0,content_39d7361b4945d504,77.0,0.0,0.000000,4.311688,keyword article,informational,True,False
1,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.049866,keyword article,informational,True,False
2,content_e689bc511192751a,61.0,0.0,0.000000,5.885246,keyword article,commercial,True,False
3,content_7dbc094b799e05a4,705.0,1.0,0.141844,5.863830,keyword article,informational,True,False
4,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.360000,keyword article,informational,True,False


Rows before join: 176738
Rows after join: 176738
Join preserved row count (no row multiplication): True
Raw daily rows excluded by gsc_data_available IS TRUE: 6230317
Excluded row percentage: 63.31
Pages with total_impressions = 0 (CTR remains NULL): 0
Calculation notes:
1. Impressions and clicks are summed over the month.
2. CTR = monthly total clicks / monthly total impressions x 100, not an average of daily CTRs; high-volume days therefore contribute proportionally more.
3. Average position = SUM(gsc_sum_position) / SUM(gsc_impressions); days with more impressions carry more weight.
4. Only rows where gsc_data_available IS TRUE were used.
5. dim_content was joined with LEFT JOIN on client_hash_id + content_hash_id; unmatched pages are retained and counted.
6. is_published and is_deleted are displayed, not used as filters; these fields describe the July 2026 export, not March 2026.


**2- counts and data span**

In [10]:

# Cell 3 - Counts on the raw daily March partition
stats = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates,
    DATE_DIFF('day', MIN(report_date), MAX(report_date)) + 1 AS date_span_days
FROM fact_mar
""").df().iloc[0]

for name, value in stats.items():
    print(f"{name}: {value}")

unique_grain = con.sql("""
SELECT COUNT(*) FROM (
    SELECT DISTINCT report_date, client_hash_id, content_hash_id
    FROM fact_mar
)
""").fetchone()[0]

multi_client_contents = con.sql("""
SELECT COUNT(*) FROM (
    SELECT content_hash_id
    FROM fact_mar
    WHERE content_hash_id IS NOT NULL
    GROUP BY content_hash_id
    HAVING COUNT(DISTINCT client_hash_id) > 1
)
""").fetchone()[0]

page_rows = con.sql("SELECT COUNT(*) FROM page_month").fetchone()[0]
page_distinct_contents = con.sql("""
SELECT COUNT(DISTINCT content_hash_id) FROM page_month
""").fetchone()[0]

print("Extra grain checks:")
print("Distinct (report_date, client, content) combinations:", unique_grain)
print("Distinct combinations equal total raw rows:", unique_grain == stats["total_rows"])
print("Content IDs appearing under more than one client (count only):", multi_client_contents)
print("page_month row count:", page_rows)
print("Distinct content IDs in page_month:", page_distinct_contents)
print("page_month rows equal distinct content IDs:", page_rows == page_distinct_contents)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_rows: 9841378
distinct_clients: 55
distinct_contents: 331437
min_date: 2026-03-01 00:00:00
max_date: 2026-03-31 00:00:00
distinct_dates: 31
date_span_days: 31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extra grain checks:
Distinct (report_date, client, content) combinations: 9841378
Distinct combinations equal total raw rows: True
Content IDs appearing under more than one client (count only): 0
page_month row count: 176738
Distinct content IDs in page_month: 176738
page_month rows equal distinct content IDs: True


**3- Missing values**

In [11]:

# Cell 4 - Missing values and availability

# A. Availability on raw daily rows
availability = con.sql("""
SELECT
    COUNT(*) AS n,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS yes,
    COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS no,
    COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS unknown
FROM fact_mar
""").df().iloc[0]

n = int(availability["n"])
for label, key in [
    ("IS TRUE", "yes"), ("IS FALSE", "no"), ("IS NULL", "unknown")
]:
    count = int(availability[key])
    print(f"A - gsc_data_available {label}: {count} ({100*count/n:.2f}%)" if n else f"A - {label}: {count} (N/A)")

# B. Pages without a dimension match
b = con.sql("""
SELECT COUNT(*) AS n,
       COUNT(*) FILTER (WHERE matched_in_dim IS FALSE) AS missing
FROM page_month
""").df().iloc[0]
print("B - Pages unmatched in dim_content:", int(b["missing"]),
      f"({100*b['missing']/b['n']:.2f}%)" if b["n"] else "(N/A)")

# C. Missing or blank metadata among matched pages
for col in ["content_type", "main_intent", "is_published", "is_deleted"]:
    row = con.sql(f"""
        SELECT COUNT(*) AS n,
               COUNT(*) FILTER (
                   WHERE {col} IS NULL
                      OR TRIM(CAST({col} AS VARCHAR)) = ''
               ) AS missing
        FROM page_month
        WHERE matched_in_dim IS TRUE
    """).df().iloc[0]
    pct = 100 * row["missing"] / row["n"] if row["n"] else None
    print(f"C - {col} missing/blank: {int(row['missing'])} "
          f"({pct:.2f}%) among matched pages" if pct is not None
          else f"C - {col}: no matched pages")

# D1. Missing main_intent by content_type
print("D - Missing main_intent by content_type (count and within-type percentage):")
display(con.sql("""
SELECT
    COALESCE(content_type, '<NULL>') AS content_type,
    COUNT(*) AS pages,
    COUNT(*) FILTER (
        WHERE main_intent IS NULL OR TRIM(main_intent) = ''
    ) AS missing_main_intent,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE main_intent IS NULL OR TRIM(main_intent) = ''
    ) / NULLIF(COUNT(*), 0), 2) AS missing_pct
FROM page_month
GROUP BY content_type
ORDER BY pages DESC
""").df())

# D2. Missing content_type by main_intent
print("D - Missing content_type by main_intent (count and within-intent percentage):")
display(con.sql("""
SELECT
    COALESCE(main_intent, '<NULL>') AS main_intent,
    COUNT(*) AS pages,
    COUNT(*) FILTER (
        WHERE content_type IS NULL OR TRIM(content_type) = ''
    ) AS missing_content_type,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE content_type IS NULL OR TRIM(content_type) = ''
    ) / NULLIF(COUNT(*), 0), 2) AS missing_pct
FROM page_month
GROUP BY main_intent
ORDER BY pages DESC
""").df())

# D3. Client concentration, output counts and percentages only
intent_concentration = con.sql("""
WITH missing AS (
    SELECT client_hash_id
    FROM page_month
    WHERE main_intent IS NULL OR TRIM(main_intent) = ''
), by_client AS (
    SELECT client_hash_id, COUNT(*) AS n
    FROM missing
    GROUP BY client_hash_id
)
SELECT
    (SELECT COUNT(DISTINCT client_hash_id) FROM missing) AS clients,
    (SELECT COUNT(*) FROM missing) AS pages,
    COALESCE((SELECT MAX(n) FROM by_client), 0) AS largest_client_pages
""").df().iloc[0]

mi_pages = int(intent_concentration["pages"])
mi_largest = int(intent_concentration["largest_client_pages"])
print("D - Clients with at least one missing main_intent:",
      int(intent_concentration["clients"]))
print("D - Missing-main_intent pages belonging to the largest client:",
      f"{100*mi_largest/mi_pages:.2f}%" if mi_pages else "N/A")

unmatched_concentration = con.sql("""
WITH missing AS (
    SELECT client_hash_id
    FROM page_month
    WHERE matched_in_dim IS FALSE
), by_client AS (
    SELECT client_hash_id, COUNT(*) AS n
    FROM missing
    GROUP BY client_hash_id
)
SELECT
    (SELECT COUNT(DISTINCT client_hash_id) FROM missing) AS clients,
    (SELECT COUNT(*) FROM missing) AS pages,
    COALESCE((SELECT MAX(n) FROM by_client), 0) AS largest_client_pages
""").df().iloc[0]

unmatched_pages = int(unmatched_concentration["pages"])
unmatched_largest = int(unmatched_concentration["largest_client_pages"])
print("D - Clients with at least one unmatched page:",
      int(unmatched_concentration["clients"]))
print("D - Unmatched pages belonging to the largest client:",
      f"{100*unmatched_largest/unmatched_pages:.2f}%"
      if unmatched_pages else "N/A")

# E. Low-volume context; do not filter these pages out
volume = con.sql("""
SELECT COUNT(*) AS total,
       COUNT(*) FILTER (WHERE total_impressions < 100) AS low_volume
FROM page_month
""").df().iloc[0]

low_n = int(volume["low_volume"])
all_n = int(volume["total"])
print("E - Pages with total_impressions < 100:", low_n,
      f"({100*low_n/all_n:.2f}% of all pages)" if all_n else "(N/A)")

# Summary sentences based on the measured results
print(f"A shows the distribution of TRUE, FALSE and NULL availability across {n} raw daily rows.")
print(f"B shows that {int(b['missing'])} of {int(b['n'])} aggregated pages have no matching dimension row.")
print("C shows the missing/blank metadata rates among pages that matched dim_content.")
print("D shows whether missing metadata varies by content category or is concentrated among particular clients; client identifiers are not displayed.")


A - gsc_data_available IS TRUE: 3611061 (36.69%)
A - gsc_data_available IS FALSE: 6230317 (63.31%)
A - gsc_data_available IS NULL: 0 (0.00%)
B - Pages unmatched in dim_content: 0 (0.00%)
C - content_type missing/blank: 0 (0.00%) among matched pages
C - main_intent missing/blank: 15839 (8.96%) among matched pages
C - is_published missing/blank: 0 (0.00%) among matched pages
C - is_deleted missing/blank: 0 (0.00%) among matched pages
D - Missing main_intent by content_type (count and within-type percentage):


,content_type,pages,missing_main_intent,missing_pct
0,keyword article,159906,2363,1.48
1,feedly article,13476,13476,100.00
2,comparison article,3356,0,0.00


D - Missing content_type by main_intent (count and within-intent percentage):


,main_intent,pages,missing_content_type,missing_pct
0,informational,100992,0,0.0
1,transactional,31617,0,0.0
2,commercial,27775,0,0.0
3,<NULL>,15839,0,0.0
4,navigational,515,0,0.0


D - Clients with at least one missing main_intent: 29
D - Missing-main_intent pages belonging to the largest client: 69.99%
D - Clients with at least one unmatched page: 0
D - Unmatched pages belonging to the largest client: N/A
E - Pages with total_impressions < 100: 75297 (42.60% of all pages)
A shows the distribution of TRUE, FALSE and NULL availability across 9841378 raw daily rows.
B shows that 0 of 176738 aggregated pages have no matching dimension row.
C shows the missing/blank metadata rates among pages that matched dim_content.
D shows whether missing metadata varies by content category or is concentrated among particular clients; client identifiers are not displayed.


**4- Windows**

In [13]:

# Cell 5 - Time window
window = con.sql("""
SELECT
    MIN(CAST(report_date AS DATE)) AS start_date,
    MAX(CAST(report_date AS DATE)) AS end_date,
    COUNT(DISTINCT CAST(report_date AS DATE)) AS days_present
FROM fact_mar
""").df().iloc[0]

start_date = window["start_date"]
end_date = window["end_date"]
days_present = int(window["days_present"])

span = con.sql("""
SELECT DATE_DIFF(
    'day',
    MIN(CAST(report_date AS DATE)),
    MAX(CAST(report_date AS DATE))
) + 1
FROM fact_mar
""").fetchone()[0]

full_month = (
    str(start_date) == "2026-03-01"
    and str(end_date) == "2026-03-31"
    and days_present == 31
)

missing_dates = con.sql("""
SELECT CAST(day AS DATE) AS missing_date
FROM generate_series(
    DATE '2026-03-01',
    DATE '2026-03-31',
    INTERVAL 1 DAY
) AS t(day)
WHERE CAST(day AS DATE) NOT IN (
    SELECT DISTINCT CAST(report_date AS DATE)
    FROM fact_mar
    WHERE report_date IS NOT NULL
)
ORDER BY missing_date
""").df()["missing_date"].astype(str).tolist()

print("Window start date found:", start_date)
print("Window end date found:", end_date)
print("Distinct dates present:", days_present)
print("Date span in days (max - min + 1):", span)
print("Full calendar month (2026-03-01 to 2026-03-31, 31 days):", full_month)
print("Expected calendar days with no rows:", len(missing_dates))
print("Missing dates:", missing_dates if missing_dates else "None")
print("Month partition used:", MONTH)
print("All queries in Cells 1 to 5 were run only on month=2026-03; no other fact table month was scanned.")


Window start date found: 2026-03-01 00:00:00
Window end date found: 2026-03-31 00:00:00
Distinct dates present: 31
Date span in days (max - min + 1): 31
Full calendar month (2026-03-01 to 2026-03-31, 31 days): False
Expected calendar days with no rows: 0
Missing dates: None
Month partition used: 2026-03
All queries in Cells 1 to 5 were run only on month=2026-03; no other fact table month was scanned.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.